# Persona drift along the Assistant Axis, read through the Jacobian lens

Reproduces the figures from `run.py` outputs. Run the pipeline first, e.g.

```bash
python experiments/persona_drift/run.py --model Qwen/Qwen2.5-7B-Instruct --out out
```

Set `OUT` below to the run's output directory.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt

%matplotlib inline

HERE = Path.cwd()
if not (HERE / "analysis.py").exists():
    HERE = Path("experiments/persona_drift")
sys.path.insert(0, str(HERE))
import analysis  # noqa: E402

OUT = Path("out")  # run.py --out
meta, rs = analysis.load(OUT)
HIT = meta["hit_rank"]
print(
    meta["model"],
    "| band",
    meta["band"],
    "| steer layer",
    meta["steer_layer"],
    "| scale",
    meta["scale_mode"],
    round(meta["scale"], 2),
)
print(len(rs), "transcripts")

## Transcripts per drift strength

In [ ]:
tr = json.loads((OUT / "transcripts.json").read_text())
for r in tr["records"]:
    if r["probe_id"] == "identity_0" or r["category"] == "organic":
        print(
            f"--- {r['condition']} ({r['probe_id']})\n{r['conversation'][-1]['content'][:300]}\n"
        )

## (a) Sanity check: drift strength vs axis projection of the mean response residual

In [ ]:
analysis.fig_projection(meta, rs)
plt.show()

## (b) Persona / assistant token sets in the workspace band vs s and response position

In [ ]:
analysis.fig_band_heatmaps(meta, rs, HIT)
plt.tight_layout()
plt.show()
analysis.fig_band_hit_vs_position(meta, rs, HIT, "persona")
plt.show()
analysis.fig_band_hit_vs_position(meta, rs, HIT, "assistant")
plt.show()

## (c) Slice pages (one per condition)

In [ ]:
from jlens.vis import notebook_iframe

pages = sorted((OUT / "slices").glob("*.html"))
print([p.name for p in pages])
notebook_iframe(pages[0].read_text()) if pages else None

## (d) Where the axis projection changes vs where persona content becomes lens-readable

In [ ]:
analysis.fig_layer_profiles(meta, rs, HIT)
plt.tight_layout()
plt.show()
print(json.dumps(analysis.onset_layers(meta, rs, HIT), indent=1))

## Organic multi-turn drift

In [ ]:
analysis.fig_organic(meta, rs, HIT)
plt.tight_layout()
plt.show()

## Does persona content reach the band before it is emitted?

In [ ]:
per_s = analysis.per_s_metrics(meta, rs, HIT)
keys = [
    "proj_steer_layer",
    "persona_band_hit_rate",
    "persona_output_hit_rate",
    "persona_emitted_rate",
    "persona_frac_band_hit",
    "persona_frac_emitted",
    "persona_frac_band_before_emit",
    "persona_median_lead_tokens",
    "assistant_band_hit_rate",
    "assistant_emitted_rate",
]
print("s".rjust(6), *[k[:22].rjust(23) for k in keys])
for s, m in per_s.items():
    print(
        f"{s:+6g}",
        *[(f"{m[k]:23.3f}" if m[k] is not None else "n/a".rjust(23)) for k in keys],
    )